In [16]:
!pip install -q langchain-groq langgraph pymupdf langchain_community langchain_chroma

In [2]:
!pip install -q "unstructured[pdf]"
!apt install poppler-utils tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.9/527.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━

In [2]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
import os
from uuid import uuid4

from unstructured.documents.elements import NarrativeText, Title
import fitz
import re
from unstructured.partition.pdf import partition_pdf


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b")
llm_summariser = ChatGroq(model="llama-3.1-8b-instant")

In [3]:
def find_toc_pages(doc, search_limit=20):
    toc_pages = []
    for i in range(min(len(doc), search_limit)):
        page = doc[i]
        text = page.get_text("text", sort=True).lower()
        if "contents" in text or "table of contents" in text:
            toc_pages.append(i)
    return toc_pages

def toc_raw_to_hierarchy(toc_raw):
    messages = [SystemMessage("""
        You are a document hierarchy generator. Your task is to generate a python dictionary.
        You are given the raw text on the table of contents page of a document.
        analyze it properly and structure it into a neat json object.
        The document hierarchy is only needed till depth level 2. the schema of the json object is as follows
        {
            "section_name": str
            "section_children": list[str]
        }
        You must follow this schema.
        YOur output should be json object and it will be evaluated as it is so do not give boilerplate text or backticks
        """)]
    messages.append(HumanMessage("here is the raw text on the table of contents page of the document"))
    messages.append(HumanMessage(toc_raw))

    res = llm.with_structured_output(method="json_mode").invoke(messages)
    return res

def toc_to_hierarchy(toc):
    hierarchy = []
    for i in range(len(toc)):
        if toc[i][0] == 1:
            new_section = {
                "section_name": toc[i][1],
                "section_children": []
            }
            new_section_children = []
            for j in range(i+1, len(toc)):
                if toc[j][0] == 2:
                    new_section_children.append(toc[j][1])
                elif toc[j][0] == 1:
                    break
            new_section["section_children"] = new_section_children
            hierarchy.append(new_section)
    return hierarchy

def get_doc_hierarchy(pdf_path):
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()
    toc_pages = find_toc_pages(doc)
    if toc:
        return toc_to_hierarchy(toc), toc_pages


    if toc_pages:
        toc_raw = ""
        for i in toc_pages:
            toc_raw += doc[i].get_text("text")
        return toc_raw_to_hierarchy(toc_raw), toc_pages

    return None


In [4]:
elements = partition_pdf(
    filename="report.pdf",
    strategy="fast"
    )

In [5]:
doc_hierarchy, toc_pages = get_doc_hierarchy("report.pdf")

In [6]:
current_section_idx = 0
current_section_chunks = []

if toc_pages:
    last_toc_page = toc_pages[-1]
    split_idx = next(
        (i for i, x in enumerate(elements) if x.metadata.page_number == last_toc_page + 2),
        None
    )
    elements = elements[split_idx:]

next_section_name = (
    doc_hierarchy[current_section_idx + 1]["section_name"]
    if current_section_idx + 1 < len(doc_hierarchy)
    else None
)

for e in elements:
    if next_section_name and isinstance(e, Title) and next_section_name in e.text:
        doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks
        current_section_chunks = [e]
        current_section_idx += 1
        next_section_name = (
            doc_hierarchy[current_section_idx + 1]["section_name"]
            if current_section_idx + 1 < len(doc_hierarchy)
            else None
        )
    else:
        current_section_chunks.append(e)

doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks

In [39]:
def get_level2_sections(doc_hierarchy):
    leaf_sections = []
    for section in doc_hierarchy:
        if len(section["section_children"]) == 0:
            leaf_sections.append({
                "section_id": str(uuid4()),
                "section_name": section["section_name"],
                "section_chunks": section["section_chunks"]
            })
        else:
            current_subsec_chunks = []
            split_idx = 0
            for i in range(len(section["section_chunks"])):
                e = section["section_chunks"][i]
                if isinstance(e, Title) and section["section_children"][0] in e.text:
                    leaf_sections.append({
                        "section_id": str(uuid4()),
                        "section_name": section["section_name"],
                        "section_chunks": current_subsec_chunks
                    })
                    split_idx = i
                    break
                else:
                    current_subsec_chunks.append(e)

            section["section_chunks"] = section["section_chunks"][split_idx:]
            current_subsec_idx = 0
            current_subsec_chunks = []
            next_subsec_name = (
                section["section_children"][current_subsec_idx + 1]
                if current_subsec_idx + 1 < len(section["section_children"])
                else None
            )
            for e in section["section_chunks"]:
                if next_subsec_name and isinstance(e, Title) and next_subsec_name in e.text:
                    leaf_sections.append({
                        "section_id": str(uuid4()),
                        "section_name": section["section_children"][current_subsec_idx],
                        "section_chunks": current_subsec_chunks
                    })

                    current_subsec_chunks = [e]
                    current_subsec_idx += 1
                    next_subsec_name = (
                        section["section_children"][current_subsec_idx + 1]
                        if current_subsec_idx + 1 < len(section["section_children"])
                        else None
                    )
                else:
                    current_subsec_chunks.append(e)

            leaf_sections.append({
                "section_id": str(uuid4()),
                "section_name": section["section_children"][current_subsec_idx],
                "section_chunks": current_subsec_chunks
            })

    return leaf_sections

In [40]:
leaf_sections = get_level2_sections(doc_hierarchy)

In [41]:
leaf_sections

[{'section_id': 'cd112822-682e-4e1f-87d0-eddb2b6e1048',
  'section_name': 'Introduction',
  'section_chunks': [<unstructured.documents.elements.Text at 0x7817e94c1d90>,
   <unstructured.documents.elements.NarrativeText at 0x7817e95ad790>]},
 {'section_id': 'b5d4e377-4b18-451b-acd4-cabad2bdbeb7',
  'section_name': 'Background',
  'section_chunks': [<unstructured.documents.elements.Title at 0x7817e95adca0>,
   <unstructured.documents.elements.NarrativeText at 0x7817e95add00>]},
 {'section_id': '18c2ad5d-d3d4-45b1-9b99-289fca5979b6',
  'section_name': 'Model Architecture',
  'section_chunks': []},
 {'section_id': '15f49ea4-07e0-479b-8aa0-0a56cacf3ad2',
  'section_name': 'Encoder and Decoder Stacks',
  'section_chunks': [<unstructured.documents.elements.Title at 0x7817e955bd70>,
   <unstructured.documents.elements.NarrativeText at 0x7817e95668a0>]},
 {'section_id': '1994f02a-aca7-45fe-b9b8-b93f4f1706a1',
  'section_name': 'Attention',
  'section_chunks': [<unstructured.documents.elements.T

In [27]:
def get_summary(text):
    messages = [SystemMessage("""
    Your job is to summarize the given raw text. Chunks of raw text are given to you and you have to analyze properly what all is contained in them.
    Pay special attention to table captions, image captions and formulas and include them in your summaries.
    You can skip the factual numeric details but you must tell what all is contained in the given chunk of raw text.

    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(f"Here is the raw text: {text}"))
    res = llm_summariser.invoke(messages)

    return res.content

In [42]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500)

summaries = {}

for sec in leaf_sections:
    sec_text = []
    for sec_chunk in sec["section_chunks"]:
        if isinstance(sec_chunk, NarrativeText):
            sec_text.append(sec_chunk.text)
    if len(sec_text) == 0:
        continue
    raw_sec_text = "\n".join(sec_text)
    sec_text_chunks = text_splitter.split_text(raw_sec_text)
    sec_summaries = []
    for sec_chunk in sec_text_chunks:
        sec_summaries.append(get_summary(sec_chunk))

    if len(sec_summaries) > 1:
        sec_summary = get_summary("\n".join(sec_summaries))
    else:
        sec_summary = sec_summaries[0]

    summaries[sec["section_id"]] = sec_summary

In [44]:
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

heading_store = Chroma(collection_name="heading_store", embedding_function=embeddings)

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "section_id"

# The retriever (empty to start)
heading_retriever = MultiVectorRetriever(
    vectorstore=heading_store,
    docstore=store,
    id_key=id_key,
)

In [50]:
heading_retriever.vectorstore.add_documents([
    Document(
        page_content=sec_sum,
        metadata={id_key: sec_id}
    )
    for sec_id, sec_sum in summaries.items()
])
heading_retriever.docstore.mset([
    (sec["section_id"], sec)
    for sec in leaf_sections
])

In [67]:
docs = heading_retriever.invoke("What is the per layer complexity?")

In [ ]:
# {
#     "section_id": {
#         "high_res_chunks": any,
#         "image_store": [{
#             "image_id": str,
#             "image_data": str,
#             "image_summary": str
#         },...],
#         "table_store": [{
#             "table_id": str,
#             "table_data": str,
#             "table_summary": str
#         },...],
#         "text_store": [{
#             "text_id": str,
#             "text_data": str,
#         },...],

#     },...
# }

In [69]:
section_cache = {}

In [ ]:
for doc in docs:
    section_id = doc["section_id"]
    if section_id not in section_cache:
        continue
        ### create high res chunks, image summary, table summary etc and store in cache

In [72]:
# use sections in docs and section cache to create vector index of reqd sections.
# retrieve docs from that vector index
# send docs to evaluator llm, check if new vector index need or new docs from old index needed